In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from glob import glob
import pandas as pd
import pickle as pkl

c:\Users\debot\micromamba\envs\info-decom-mirco\Lib\site-packages\scikits\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [3]:
def process_animal_file(file_path, animal_name):
    """
    Reads a pickle file and extracts balanced accuracy and null distribution means.
    """
    with open(file_path, "rb") as f:
        data = pkl.load(f)

    records = []
    for kf, regions_dict in data.items():
        for region_pair, content in regions_dict.items():
            dec = content["decoder_results"]
            nulls = content["null_results"]

            acc_a = dec.get("balanced_acc_A")
            acc_b = dec.get("balanced_acc_B")

            acc_a_cong = dec.get("balanced_acc_A_cong")
            acc_a_incong = dec.get("balanced_acc_A_incong")
            acc_b_cong = dec.get("balanced_acc_B_cong")
            acc_b_incong = dec.get("balanced_acc_B_incong")

            parts = region_pair.rsplit("_", 1)
            label_a, label_b = [p.replace("['", "").replace("']", "") for p in parts]

            records.append(
                {
                    "animal": animal_name,
                    "key_frame": kf,
                    "pair": region_pair,
                    "region_label": label_a,
                    "accuracy": dec.get("balanced_acc_A"),
                    "null_mean": np.mean([n.get("balanced_acc_A", 0) for n in nulls]),
                    "pos": "First",
                    "cong": acc_a_cong,
                    "incong": acc_a_incong,
                }
            )
            records.append(
                {
                    "animal": animal_name,
                    "key_frame": kf,
                    "pair": region_pair,
                    "region_label": label_b,
                    "accuracy": dec.get("balanced_acc_B"),
                    "null_mean": np.mean([n.get("balanced_acc_B", 0) for n in nulls]),
                    "pos": "Second",
                    "cong": acc_a_cong,
                    "incong": acc_a_incong,
                }
            )
    return records


def plot_data(df, target_frame=None):

    if target_frame == None:
        target_frame = 1

    df_subset = df[df["key_frame"] == target_frame]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    color = (["#FDBF6F", "#A6CEE3"],)

    sns.barplot(
        data=df_subset,
        x="region_label",
        y="accuracy",
        ax=axes[0],
        palette="viridis",
        errorbar="sd",
        capsize=0.1,
    )

    null_means = df_subset.groupby("region_label")["null_mean"].mean()
    regions = [t.get_text() for t in axes[0].get_xticklabels()]

    for i, region in enumerate(regions):
        if region in null_means:
            null_val = null_means[region]

            axes[0].hlines(
                y=null_val,
                xmin=i - 0.4,
                xmax=i + 0.4,
                colors="red",
                linestyles="--",
                linewidth=2,
                label="Null Mean" if i == 0 else "",
            )

    axes[0].set_title(f"Mean Accuracy per Region")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_xlabel("Region Label")

    handles, labels = axes[0].get_legend_handles_labels()

    if labels:
        axes[0].legend()
    else:

        from matplotlib.lines import Line2D

        custom_lines = [Line2D([0], [0], color="red", lw=2, linestyle="--")]
        axes[0].legend(custom_lines, ["Null Performance"])

    plot2_data = df_subset.melt(
        id_vars="region_label",
        value_vars=["cong", "incong"],
        var_name="Condition",
        value_name="Score",
    )
    custom_palette = {"incong": "#FDBF6F", "cong": "#A6CEE3"}
    sns.barplot(
        data=plot2_data,
        x="region_label",
        y="Score",
        hue="Condition",
        ax=axes[1],
        palette=custom_palette,
        errorbar="sd",
        capsize=0.1,
    )

    axes[1].set_title(f"Congruent vs Incongruent Accuracy")
    axes[1].set_ylabel("Score")
    axes[1].set_xlabel("Region Label")
    axes[1].legend(title="Condition")

    plt.tight_layout()
    plt.show()

In [4]:
stim_files = glob("../data/generated/wfi_decoders_entire_data/allregions/stim/*.pkl")
choice_files = glob("../data/generated/wfi_decoders_entire_data/allregions/choice/*.pkl")

In [5]:
all_records_stim = []
for name, path in enumerate(stim_files):
    all_records_stim.extend(process_animal_file(path, name))
df_stim = pd.DataFrame(all_records_stim)

all_records_choice = []
for name, path in enumerate(choice_files):
    all_records_choice.extend(process_animal_file(path, name))
df_choice = pd.DataFrame(all_records_choice)

In [13]:
# pick only performances that are better than null

In [14]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

In [86]:
def get_p_value(row):
    n_better = np.sum(np.asarray(row["null_mean"]) >= row["accuracy"])
    return (n_better + 1) / (len(row["null_mean"]) + 1)

In [87]:
# run this for any df


def find_significant_sessions(df):

    df["p_val"] = df.apply(get_p_value, axis=1)

    significant_sessions = df.groupby(["animal", "key_frame", "pair"]).filter(
        lambda x: (x["p_val"] < 0.05).all()
    )

    final_list = significant_sessions.pivot_table(
        index=["animal", "key_frame", "pair"],
        columns="pos",
        values="region_label",
        aggfunc="first",  # Just grabs the region name string
    ).reset_index()

    # Clean up column names
    final_list.columns.name = None  # Remove the 'pos' label from the header
    final_list = final_list.rename(columns={"First": "region_1", "Second": "region_2"})

    # Display
    print(f"Total unique pairs: {len(final_list)}")

    return final_list

In [88]:
significant_choice = find_significant_sessions(df=df_choice)

Total unique pairs: 9526


In [89]:
significant_stim = find_significant_sessions(df=df_stim)

Total unique pairs: 2890


In [ ]:
significant_choice[
    (significant_choice["key_frame"] == 2)
    & (significant_choice["animal"] == "12046ccd-b737-4c01-bc70-400f3b69291b")
]

,animal,key_frame,pair,region_1,region_2
253,12046ccd-b737-4c01-bc70-400f3b69291b,2,['MOB']_['MOp'],MOB,MOp
254,12046ccd-b737-4c01-bc70-400f3b69291b,2,['MOB']_['MOs'],MOB,MOs
255,12046ccd-b737-4c01-bc70-400f3b69291b,2,['MOB']_['RSPagl'],MOB,RSPagl
256,12046ccd-b737-4c01-bc70-400f3b69291b,2,['MOB']_['RSPd'],MOB,RSPd
257,12046ccd-b737-4c01-bc70-400f3b69291b,2,['MOB']_['SSp-bfd'],MOB,SSp-bfd
...,...,...,...,...,...
381,12046ccd-b737-4c01-bc70-400f3b69291b,2,['VISam']_['VISpm'],VISam,VISpm
382,12046ccd-b737-4c01-bc70-400f3b69291b,2,['VISam']_['VISrl'],VISam,VISrl
383,12046ccd-b737-4c01-bc70-400f3b69291b,2,['VISp']_['VISpm'],VISp,VISpm
384,12046ccd-b737-4c01-bc70-400f3b69291b,2,['VISp']_['VISrl'],VISp,VISrl


In [95]:
with open(
    "../data/generated/wfi_decoders_entire_data/allregions/choice/1efa20cb-e3e9-4a28-b4f0-779e14a67d47_wfi_choiceentire_data.pkl",
    "rb",
) as f:
    x = pkl.load(f)

In [100]:
x[2]["['MOB']_['MOp']"]["decoder_results"].keys()

dict_keys(['accuracy_A', 'accuracy_B', 'balanced_acc_A', 'balanced_acc_B', 'probs_A', 'probs_B', 'y_true', 'y_cong', 'probs_A_cong', 'probs_B_cong', 'balanced_acc_A_cong', 'y_incong', 'probs_A_incong', 'probs_B_incong', 'balanced_acc_A_incong'])

In [107]:
pairs = significant_choice[
    (significant_choice["key_frame"] == 2)
    & (significant_choice["animal"] == "12046ccd-b737-4c01-bc70-400f3b69291b")
]["pair"].values

In [ ]:
x[2][pairs[0]]["decoder_results"]